# Low-Light Image Enhancement with an AI Assistant

## Step 1: Load an image and measure how dark it is

Before we enhance anything, the program must first *understand* the image.
In this step we measure brightness, contrast and noise, and turn the
brightness into a simple label (Normal, Dim, Dark, Very dark).
Later, Jarvis will use this label to decide which enhancement method to use.

In [ ]:
# Step 1.1 - Import the libraries we need
import cv2                       # OpenCV: reads and processes images
import numpy as np               # NumPy: fast maths on image arrays
import matplotlib.pyplot as plt  # Matplotlib: shows images and graphs

In [ ]:
# Step 1.2 - Load an image
# "sample"  -> uses a built-in test photo that we darken ourselves
# "upload"  -> lets you pick your own dark photo from your computer
IMAGE_SOURCE = "sample"

if IMAGE_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()                # a file picker opens
    file_name = list(uploaded.keys())[0]
    img = cv2.imread(file_name)              # OpenCV loads images in BGR order
else:
    from skimage import data
    bright = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)
    # Make it dark: raising pixel values (scaled 0-1) to a power above 1 darkens them
    img = ((bright / 255.0) ** 2.5 * 0.4 * 255).astype(np.uint8)

print("Image size (height, width, channels):", img.shape)

In [ ]:
# Step 1.3 - Functions that measure the image

def to_gray(image):
    # Brightness is easiest to measure on a single grey channel
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

def get_brightness(image):
    # Average pixel value scaled to 0-1 (0 = black, 1 = white)
    return to_gray(image).mean() / 255.0

def get_contrast(image):
    # Standard deviation of pixel values: how spread out the tones are
    return to_gray(image).std() / 255.0

def get_dark_ratio(image):
    # Fraction of pixels that are very dark (below 15% brightness)
    return (to_gray(image) < 0.15 * 255).mean()

def get_noise(image):
    # Noise estimate (Immerkaer's method): a filter that ignores smooth areas
    # and responds mostly to random grain
    gray = to_gray(image).astype(np.float32)
    h, w = gray.shape
    kernel = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], dtype=np.float32)
    filtered = cv2.filter2D(gray, -1, kernel)
    return np.sum(np.abs(filtered)) * np.sqrt(0.5 * np.pi) / (6.0 * (w - 2) * (h - 2))

def darkness_level(brightness):
    # Turn the brightness number into a label Jarvis can use later
    if brightness >= 0.40:
        return "Normal"
    if brightness >= 0.25:
        return "Dim"
    if brightness >= 0.12:
        return "Dark"
    return "Very dark"

def analyze_image(image):
    brightness = get_brightness(image)
    level = darkness_level(brightness)
    return {
        "Brightness (0-1)": round(brightness, 3),
        "Contrast (0-1)": round(get_contrast(image), 3),
        "Dark pixel ratio": round(get_dark_ratio(image), 3),
        "Noise level": round(get_noise(image), 2),
        "Darkness level": level,
        "Needs enhancement": level != "Normal",
    }

In [ ]:
# Step 1.4 - Analyse the image and show the result
stats = analyze_image(img)
for name, value in stats.items():
    print(f"{name:20s}: {value}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # matplotlib expects RGB
axes[0].set_title("Input image")
axes[0].axis("off")

axes[1].hist(to_gray(img).ravel(), bins=64, range=(0, 255), color="gray")
axes[1].set_title("Brightness histogram")
axes[1].set_xlabel("Pixel value (0 = black, 255 = white)")
axes[1].set_ylabel("Number of pixels")
plt.tight_layout()
plt.show()